<a href="https://colab.research.google.com/github/gyeong061/AIFFEL_QUEST_ENG/blob/main/05_LLM/LLM03/Practice_Q1_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 2 실습 — Advanced·Modular RAG + RAGAS 평가 (퀘스트 1·2 완성본)

원본의 설명과 실습 순서는 유지하고, 필수 TODO를 채웠습니다. 또한 최종 Advanced 파이프라인이 실제로
**Multi-Query → RAG-Fusion(RRF) → HyDE → Cross-Encoder Reranking → Self-RAG** 순서를 사용하도록 연결했습니다.

> 실행 전 주의: OpenAI API 비용, Hugging Face 데이터/모델 다운로드, RAGAS 평가 시간이 발생합니다.


# 들어가며

Day 1에서는 가장 기본형인 **Naive RAG** 파이프라인을 직접 구현해 보았습니다. 이번 실습에서는 한국어 QA 벤치마크 **KorQuAD v1** 데이터셋 위에서 **Advanced·Modular RAG** 의 핵심 기법(Multi-Query, RAG-Fusion, HyDE, Reranking, Self-RAG)을 단계적으로 적용하고, 그 결과를 **RAGAS** 로 정량 평가합니다.

이번 실습이 끝나면 다음을 직접 말할 수 있게 됩니다.
- Naive RAG 대비 **어떤 단계**를 보강하면 정답률이 올라가는가
- Multi-Query / RAG-Fusion / HyDE / Reranker / Self-RAG 는 각각 **어떤 코드 라인**으로 적용하는가
- RAGAS 의 4대 지표(Faithfulness · Answer Relevance · Context Precision · Context Recall)는 어떻게 계산되고 어떻게 읽는가
- 내 RAG 가 ‘얼마나 좋아졌는지’를 **숫자로** 보여주는 방법

## Step 0 : 설치와 준비  
Day 1과 동일하게 Colab에서 진행한다고 가정합니다.

In [ ]:
# Colab pre-installed langchain 0.3 / ragas 0.1~0.4 를 ragas 0.2.10 호환 조합으로 정리합니다.
# 처음 실행 시 약 3~5분 걸립니다. 진행률 출력을 보면서 기다리세요 (멈춘 게 아닙니다).

# 1) 기존 langchain / ragas 패키지 제거 — 버전 충돌로 인한 pip resolver 백트래킹 방지
!pip uninstall -y ragas ragas-experimental langchain langchain-core langchain-community langchain-openai langchain-text-splitters langchain-chroma

# 2) 0.2 시리즈 패치 버전까지 핀 설치 — resolver 부담 최소화 (-q 제거해서 진행률 보이게)
!pip install --no-cache-dir \
    "ragas==0.2.10" \
    "langchain==0.2.17" \
    "langchain-core==0.2.43" \
    "langchain-community==0.2.19" \
    "langchain-openai==0.1.25" \
    "langchain-text-splitters==0.2.4" \
    "langchain-chroma==0.1.4" \
    pypdf chromadb tiktoken sentence-transformers datasets nest_asyncio pandas

Found existing installation: ragas 0.2.10
Uninstalling ragas-0.2.10:
  Successfully uninstalled ragas-0.2.10
Found existing installation: langchain 0.2.17
Uninstalling langchain-0.2.17:
  Successfully uninstalled langchain-0.2.17
Found existing installation: langchain-core 0.2.43
Uninstalling langchain-core-0.2.43:
  Successfully uninstalled langchain-core-0.2.43
Found existing installation: langchain-community 0.2.19
Uninstalling langchain-community-0.2.19:
  Successfully uninstalled langchain-community-0.2.19
Found existing installation: langchain-openai 0.1.25
Uninstalling langchain-openai-0.1.25:
  Successfully uninstalled langchain-openai-0.1.25
Found existing installation: langchain-text-splitters 0.2.4
Uninstalling langchain-text-splitters-0.2.4:
  Successfully uninstalled langchain-text-splitters-0.2.4
Found existing installation: langchain-chroma 0.1.4
Uninstalling langchain-chroma-0.1.4:
  Successfully uninstalled langchain-chroma-0.1.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

> ⚠️ **위 설치 셀(Step 0)을 실행한 뒤 반드시 [런타임 > 세션 다시 시작 (Restart session)]을 한 번 눌러주세요.**
>
> 이 셀은 Colab에 기본 설치된 langchain을 제거하고 `0.2.x` / `ragas 0.2.10` 조합으로 다운그레이드합니다. 이미 메모리에 로드된 패키지를 교체하는 것이라 Colab이 재시작을 요구합니다.
>
> 재시작 후에는 **설치 셀은 다시 실행하지 말고** 이 셀 아래(키 설정)부터 순서대로 실행하면 됩니다.

In [ ]:
import os
# chromadb 익명 통계 전송 끄기 — posthog SDK 인자 충돌로 ERROR 로그가 뜨는 것 방지
os.environ["ANONYMIZED_TELEMETRY"] = "False"

import nest_asyncio
nest_asyncio.apply()  # RAGAS가 Colab의 비동기 이벤트 루프와 충돌하지 않도록

In [ ]:
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_KEY')

## Step 1 : KorQuAD v1 위에서 Naive RAG 베이스라인 만들기

Day 1에서 만든 RAG 파이프라인을 한국어 QA 벤치마크 **KorQuAD v1** 위에 다시 한 번 올립니다. 이후 단계는 모두 이 베이스라인 위에 ‘덧붙이는’ 방식입니다.

- HuggingFace `datasets` 로 KorQuAD v1 자동 다운로드 (별도 PDF 업로드 불필요)
- 일부만 샘플링해 토큰 비용 통제 (unique context 약 200개)
- Embedding → VectorStore → Retriever → LLM
- 검색 전략은 단순 `similarity` (top-k)

**📥 데이터셋**: <https://huggingface.co/datasets/KorQuAD/squad_kor_v1>

In [ ]:
from datasets import load_dataset
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
import tiktoken

tokenizer = tiktoken.get_encoding("cl100k_base")

def tiktoken_len(text):
    return len(tokenizer.encode(text))


# 1) 데이터셋 로드 + 2,000개 샘플링 + context 중복 제거
raw_ds = (
    load_dataset("squad_kor_v1", split="validation")
    .shuffle(seed=42)
    .select(range(2000))
)

unique = {}
for example in raw_ds:
    if example["context"] not in unique:
        unique[example["context"]] = example["title"]

context_docs = [
    Document(page_content=context, metadata={"title": title, "domain": "KorQuAD"})
    for context, title in unique.items()
]

# 2) 문서를 토큰 기준 Chunk로 한 번만 분할
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=tiktoken_len,
)
docs = splitter.split_documents(context_docs)

# 3) KorQuAD 전용 Chroma collection에 배치 적재
embedding = OpenAIEmbeddings(model="text-embedding-3-small")
db = Chroma(collection_name="korquad_v1", embedding_function=embedding)

BATCH = 100
for start in range(0, len(docs), BATCH):
    db.add_documents(docs[start:start + BATCH])

# 4) Naive Retriever: 질문 하나로 similarity top-3 검색
naive_retriever = db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)

# 5) 답변 생성용 LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print(
    f"베이스라인 준비 완료 — unique context: {len(context_docs)}, "
    f"chunks: {len(docs)}"
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

squad_kor_v1/train-00000-of-00001.parque(…):   0%|          | 0.00/11.6M [00:00<?, ?B/s]

squad_kor_v1/validation-00000-of-00001.p(…):   0%|          | 0.00/1.16M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60407 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5774 [00:00<?, ? examples/s]

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


베이스라인 준비 완료 — unique context: 847, chunks: 1279


베이스라인 RAG로 간단한 질의를 던져 답이 나오는지 확인합니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

RAG_PROMPT = ChatPromptTemplate.from_template(
    "다음 문서를 참고해 질문에 한국어로 간결하게 답하세요. 문서에 없는 내용은 만들지 마세요.\n\n"
    "[문서]\n{context}\n\n"
    "[질문]\n{question}\n\n"
    "[답변]"
)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

naive_chain = (
    {"context": naive_retriever | format_docs,
     "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

# 데이터셋에서 첫 질문 하나를 뽑아 테스트
TEST_Q = raw_ds[0]["question"]
print("Q:", TEST_Q)
print("A:", naive_chain.invoke(TEST_Q))

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Q: 2004년 이명박이 서울시장 재직시절 전면적으로 개선한 것은?
A: 대중교통체계입니다.


## Step 2 : Pre-retrieval 강화 — Multi-Query Retrieval  

사용자가 던진 질문 하나로만 검색하면 ‘다른 표현’으로 적힌 정답을 놓칠 수 있습니다. **Multi-Query Retrieval**은 LLM에게 ‘같은 의도의 다른 질문 N개’를 만들게 시킨 뒤, 각 질문으로 병렬 검색하고 결과를 합칩니다.

LangChain은 이를 한 클래스로 제공합니다.

In [ ]:
from langchain.retrievers.multi_query import MultiQueryRetriever
import logging
logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=db.as_retriever(search_kwargs={"k": 3}),
    llm=ChatOpenAI(model="gpt-4o-mini", temperature=0),
)

# 어떤 ‘유사 질문’으로 확장되는지 로그로 확인 가능
docs_mq = multi_query_retriever.invoke(TEST_Q)
print(f"검색된 문서 수: {len(docs_mq)}")
print("---")
print(docs_mq[0].page_content[:300])

INFO:langchain.retrievers.multi_query:Generated queries: ['2004년 이명박 서울시장이 재직할 때 어떤 주요 개선 사항이 있었나요?  ', '이명박이 2004년 서울시장으로서 추진한 주요 정책이나 변화는 무엇인가요?  ', '2004년 이명박 서울시장 재임 기간 동안 어떤 프로젝트나 계획이 전면적으로 개선되었나요?']


검색된 문서 수: 5
---
2010년 한나라당 당내경선에서 나경원, 김충환 등의 경쟁자를 물리치고, 민선 5기 지방선거에서 서울시장 재선에 도전했다. 6월 2일에 치뤄진 지방선거에서 개표 초반에 한명숙 후보에게 뒤지다가, 후반 강남 3구의 개표가 시작되면서 역전하여 민선 5기 제34대 서울특별시장으로 재선되었다. 구체적으로 강남구(+59,206, +25.68%), 서초구(+43,820, +23.66%), 송파구(+23,814, +8.19%), 강동구(+11,097, +5.33%), 용산구(+8,579, +8.24%), 양천구(+1,078, +0.51%), 영


## Step 2.5 : RAG-Fusion — Multi-Query + RRF로 묶어내기

Day2_1 노트에서 “꼭 짚고 가라”고 했던 패턴 중 하나가 **RAG-Fusion** 입니다. Step 2의 Multi-Query는 ‘유사 질문 N개로 병렬 검색’ 까지만 했는데, **RAG-Fusion** 은 그 N개 검색 결과를 **Reciprocal Rank Fusion (RRF)** 라는 간단한 공식으로 합쳐 ‘여러 쿼리에서 공통으로 상위에 떴던 문서’ 를 최상단으로 끌어올립니다.

RRF 점수 공식:

$$
\text{score}(d) = \sum_{i=1}^{N} \frac{1}{k + \text{rank}_i(d)}
$$

- $\text{rank}_i(d)$ : i번째 쿼리의 결과에서 문서 $d$ 의 순위 (1부터)
- $k$ : 스무딩 상수 (관례적으로 60)

아래 셀에서는 (1) sub-query 생성, (2) 각 sub-query 로 검색, (3) **RRF 함수는 여러분이 직접 채우기**, (4) 결과 확인까지 한 번에 해봅니다.

In [ ]:
from collections import defaultdict

# (1) sub-query 생성 — Multi-Query가 내부적으로 하는 일을 명시적으로 노출
SUBQUERY_PROMPT = ChatPromptTemplate.from_template(
    "당신은 검색 보조 AI입니다. 다음 질문과 의미는 같지만 표현이 다른 "
    "4개의 한국어 검색 쿼리를 만드세요. 오직 4개의 쿼리만 한 줄에 하나씩 "
    "출력하고, 번호나 다른 설명은 붙이지 마세요.\n\n질문: {question}"
)

def fan_out_queries(question, n=4):
    raw = (SUBQUERY_PROMPT | llm | StrOutputParser()).invoke(
        {"question": question}
    )
    queries = [line.strip() for line in raw.splitlines() if line.strip()]
    return queries[:n]


# (2) Reciprocal Rank Fusion 직접 구현
def reciprocal_rank_fusion(results_per_query, k=60, top_k=3):
    # results_per_query: 쿼리별 검색 결과 List[List[Document]]
    # k: RRF smoothing 상수. 일반적으로 60을 사용
    # top_k: 융합 후 반환할 문서 수
    scores = defaultdict(float)
    docs_by_key = {}

    for result_list in results_per_query:
        for rank, document in enumerate(result_list):
            # 같은 본문은 같은 문서로 간주해 여러 쿼리의 점수를 누적합니다.
            key = document.page_content
            scores[key] += 1.0 / (k + rank + 1)
            docs_by_key.setdefault(key, document)

    ranked = sorted(
        scores.items(),
        key=lambda item: item[1],
        reverse=True,
    )
    return [docs_by_key[key] for key, _ in ranked[:top_k]]


# (3) 한 번 돌려보기
sub_queries = fan_out_queries(TEST_Q)
print(f"확장 질문 {len(sub_queries)}개:")
for query in sub_queries:
    print(" -", query)

results_per_q = [db.similarity_search(query, k=5) for query in sub_queries]
fused = reciprocal_rank_fusion(results_per_q, k=60, top_k=3)

print("\nRAG-Fusion top-1 문서:")
print(fused[0].page_content[:300])


확장 질문 4개:
 - 2004년 이명박 서울시장 재직 중 개선한 사항은?
 - 이명박이 2004년 서울시장으로서 개선한 내용은 무엇인가?
 - 2004년 서울시장 이명박이 전면적으로 개선한 것은 어떤 것인가?
 - 이명박이 서울시장으로 재직하던 2004년에 개선한 것은 무엇인가?

RAG-Fusion top-1 문서:
2004년 서울시장 재직시절 대중교통체계를 전면적으로 개선하였다. 거리비례제를 도입하여 교통수단에 관계없이 이동한 거리에 비례해서 요금을 지불하게 바뀌면서 환승으로 인한 추가적인 교통비 부담이 없어졌다. 그 외에도 서울시 버스를 4종류로 나누고 버스 전용차로를 도로 중앙으로 옮기는 등의 많은 변화가 일시에 일어나면서 초기엔 시행착오로 인한 큰 불편을 겪기도 했다. 하지만 새 교통체계가 정착되면서 많은 긍정적인 효과를 가져오게 된다. 중앙버스차로 도입으로 버스의 평균 속도가 증가하여 정시에 도착하는 빈도가 늘어났고 환승제도로 인한 교


## Step 3 : 패턴 ② HyDE — 가상의 ‘정답’으로 진짜 정답 찾기  

질문은 짧은 의문문, 정답은 긴 평서문이라 둘의 임베딩이 의외로 멀 수 있습니다. **HyDE(Hypothetical Document Embeddings)** 는 검색 전에 LLM에게 ‘가상의 정답’을 쓰게 한 뒤, 그 가상 답변을 임베딩해서 검색합니다.

직접 구현해 보겠습니다.

In [ ]:
HYDE_PROMPT = ChatPromptTemplate.from_template(
    "당신은 해당 분야 전문가입니다. 다음 질문에 대해 그럴듯한 한국어 답변 한 문단을 작성하세요. "
    "확실하지 않다면 가장 합리적인 추측을 적어주세요.\n\n"
    "질문: {question}\n\n가상 답변:"
)

hyde_generator = HYDE_PROMPT | llm | StrOutputParser()

def hyde_retrieve(question, k=3):
    """질문 → 가상의 답변 → 가상 답변을 임베딩해 검색"""
    hypothetical = hyde_generator.invoke({"question": question})
    return db.similarity_search(hypothetical, k=k), hypothetical

docs_hyde, hyp = hyde_retrieve(TEST_Q)
print("가상 답변(HyDE):\n", hyp[:300], "\n---")
print("검색된 문서 수:", len(docs_hyde))
print("첫 문서:", docs_hyde[0].page_content[:200])

가상 답변(HyDE):
 2004년 이명박이 서울시장으로 재직하던 시절, 그는 서울시의 교통 체계를 전면적으로 개선하는 데 주력했습니다. 특히, 그는 '서울시 교통체계 개선 종합계획'을 수립하여 대중교통의 효율성을 높이고, 도로 혼잡을 줄이기 위한 다양한 정책을 시행했습니다. 이 과정에서 지하철 노선 확장과 버스 전용차선 도입, 그리고 자전거 도로 확충 등이 이루어졌으며, 이러한 노력은 서울시민의 교통 편의성을 크게 향상시키는 데 기여했습니다. 
---
검색된 문서 수: 3
첫 문서: 2004년 서울시장 재직시절 대중교통체계를 전면적으로 개선하였다. 거리비례제를 도입하여 교통수단에 관계없이 이동한 거리에 비례해서 요금을 지불하게 바뀌면서 환승으로 인한 추가적인 교통비 부담이 없어졌다. 그 외에도 서울시 버스를 4종류로 나누고 버스 전용차로를 도로 중앙으로 옮기는 등의 많은 변화가 일시에 일어나면서 초기엔 시행착오로 인한 큰 불편을 겪기도


## Step 4 : Post-retrieval 강화 — Cross-Encoder Reranking (multilingual)

검색 결과를 그대로 LLM 에 넘기지 않고, **Cross-encoder reranker** 가 (질문, 문단)을 함께 보면서 진짜 관련도를 다시 점수화합니다. 정밀도가 15~30% 개선되는 게 일반적인 보고입니다.

한국어 문서를 다루고 있으므로 다국어를 지원하는 cross-encoder 를 사용합니다. `BAAI/bge-reranker-v2-m3` 는 한국어를 포함한 100개 이상 언어에서 동작합니다. 처음 실행 시 모델 다운로드(~2GB)가 발생합니다.

In [ ]:
from sentence_transformers import CrossEncoder

# 다국어 cross-encoder (한국어 포함)
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3")

def rerank(query, docs, top_k=3):
    """검색된 docs 를 cross-encoder 로 다시 점수화해 상위 top_k 만 반환"""
    pairs = [(query, d.page_content) for d in docs]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    return [d for d, _ in ranked[:top_k]]

candidates = db.as_retriever(search_kwargs={"k": 10}).invoke(TEST_Q)
top3 = rerank(TEST_Q, candidates, top_k=3)
print(f"후보 {len(candidates)}개 → Reranker 로 상위 3개 선별")
print("최상위 문서:", top3[0].page_content[:200])

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

후보 10개 → Reranker 로 상위 3개 선별
최상위 문서: 2004년 서울시장 재직시절 대중교통체계를 전면적으로 개선하였다. 거리비례제를 도입하여 교통수단에 관계없이 이동한 거리에 비례해서 요금을 지불하게 바뀌면서 환승으로 인한 추가적인 교통비 부담이 없어졌다. 그 외에도 서울시 버스를 4종류로 나누고 버스 전용차로를 도로 중앙으로 옮기는 등의 많은 변화가 일시에 일어나면서 초기엔 시행착오로 인한 큰 불편을 겪기도


## Step 5 : Advanced RAG 체인 조립  

위에서 만든 컴포넌트들을 하나의 체인으로 묶습니다. **‘넓게 검색 → Reranker로 좁히기 → LLM 답변’** 패턴이 가장 흔히 쓰입니다.

In [ ]:
def advanced_retrieve(question, top_k=3, return_trace=False):
    # Multi-Query → RRF(+HyDE 결과) → Cross-Encoder Reranking
    # 1) 원 질문과 표현이 다른 질문들을 생성합니다.
    expanded_queries = [question, *fan_out_queries(question, n=4)]

    # 2) 각 질문으로 넓게 검색합니다.
    results_per_query = [
        db.similarity_search(query, k=5)
        for query in expanded_queries
    ]

    # 3) HyDE 가상 답변으로 찾은 결과도 하나의 검색 결과 목록으로 추가합니다.
    hyde_docs, hypothetical = hyde_retrieve(question, k=5)
    results_per_query.append(hyde_docs)

    # 4) RRF로 여러 검색 순위를 융합해 reranker 후보 10개를 만듭니다.
    fused_candidates = reciprocal_rank_fusion(
        results_per_query,
        k=60,
        top_k=10,
    )

    # 5) Cross-Encoder로 질문-문서를 함께 읽고 최종 top-k를 고릅니다.
    top_docs = rerank(question, fused_candidates, top_k=top_k)

    if return_trace:
        return top_docs, {
            "expanded_queries": expanded_queries,
            "hypothetical_document": hypothetical,
            "fused_candidates": fused_candidates,
        }
    return top_docs


def advanced_rag(question):
    top_docs = advanced_retrieve(question, top_k=3)
    answer = (RAG_PROMPT | llm | StrOutputParser()).invoke(
        {"context": format_docs(top_docs), "question": question}
    )
    return answer, top_docs


ans_adv, ctx_adv = advanced_rag(TEST_Q)
print("Advanced RAG 답변:\n", ans_adv)
print("검색된 최종 문서 수:", len(ctx_adv))


Advanced RAG 답변:
 대중교통체계입니다.
검색된 최종 문서 수: 3


## Step 5.5 : Self-RAG — 검색 필요성 판단 + 답변 자가 비평

Day2_1 노트에서 강조한 또 하나의 핵심 패턴, **Self-RAG** 입니다. Self-RAG의 핵심은 **LLM이 검색·답변 과정에 스스로 비평(critique)을 끼워 넣는다**는 점입니다.

이번 셀에서는 공식 Self-RAG 모델을 따로 받지 않고, **세 개의 작은 LLM 프롬프트**로 같은 흐름을 흉내내 봅니다.

1. **Retrieve 결정** — 질문이 들어오면, 외부 검색이 필요한지 LLM이 먼저 판단합니다. (`YES`/`NO` 한 단어)
2. **답변 생성** — `YES` 면 일반 RAG, `NO` 면 검색 없이 LLM 단독 답변.
3. **답변 자가 비평** — 생성된 답변이 컨텍스트에 충분히 근거하는지 LLM이 점검합니다. (`SUPPORTED` / `NOT_SUPPORTED`)
4. **보완 재시도** — `NOT_SUPPORTED` 면 Step 3의 **HyDE** 로 검색 쿼리를 바꿔 한 번 더 시도합니다.

코드 골격은 제공해 두었고, **두 군데 핵심 프롬프트만 여러분이 직접 채워주세요.**

In [ ]:
# Self-RAG: 검색 필요성 판단 + 답변 생성 + 자가 비평 + 재검색

# (1) 검색 필요성 판단 프롬프트 — TODO 1 완성
RETRIEVE_DECISION_PROMPT = ChatPromptTemplate.from_template(
    "다음 질문에 정확히 답하려면 외부 문서에서 구체적인 사실을 검색해야 하는지 "
    "판단하세요. 특정 인물·사건·장소·날짜·작품·기사 내용처럼 근거 확인이 필요하면 "
    "YES, 간단한 계산·일반적인 정의·일상 대화처럼 외부 문서 없이 답할 수 있으면 "
    "NO라고 답하세요. 설명이나 문장부호 없이 오직 YES 또는 NO 한 단어만 출력하세요."
    "\n\n질문: {question}"
)

# (2) 답변 자가 비평 프롬프트 — TODO 2 완성
CRITIQUE_PROMPT = ChatPromptTemplate.from_template(
    "아래 답변의 핵심 주장들이 제공된 문서 내용으로 직접 뒷받침되는지 판정하세요. "
    "모든 핵심 주장이 문서에서 확인되면 SUPPORTED, 문서에 없거나 서로 모순되는 "
    "주장이 하나라도 있으면 NOT_SUPPORTED라고 답하세요. 설명이나 문장부호 없이 "
    "오직 SUPPORTED 또는 NOT_SUPPORTED 한 단어만 출력하세요."
    "\n\n[문서]\n{context}\n\n[답변]\n{answer}"
)


def self_rag(question, max_retries=1, verbose=True):
    decision_raw = (
        RETRIEVE_DECISION_PROMPT | llm | StrOutputParser()
    ).invoke({"question": question})
    decision = decision_raw.strip().upper()

    if verbose:
        print(f"[1] Retrieve 필요? → {decision}")

    # NO가 명확할 때만 검색을 건너뜁니다. 형식이 깨지면 안전하게 검색합니다.
    if decision.startswith("NO"):
        answer = llm.invoke(question).content
        if verbose:
            print("[2] LLM 단독 답변 사용")
        return answer, []

    # 첫 시도부터 Multi-Query, RRF, HyDE, Reranking을 모두 적용합니다.
    retrieved_docs = advanced_retrieve(question, top_k=3)

    for attempt in range(max_retries + 1):
        context = format_docs(retrieved_docs)
        answer = (RAG_PROMPT | llm | StrOutputParser()).invoke(
            {"context": context, "question": question}
        )
        critique_raw = (
            CRITIQUE_PROMPT | llm | StrOutputParser()
        ).invoke({"context": context, "answer": answer})
        critique = critique_raw.strip().upper()

        supported = (
            critique.startswith("SUPPORTED")
            and "NOT_SUPPORTED" not in critique
        )

        if verbose:
            print(f"[3] 시도 {attempt + 1} — 자가 비평: {critique}")

        if supported:
            return answer, retrieved_docs

        if attempt < max_retries:
            # 실패 시 원 질문 결과와 더 넓은 HyDE 결과를 다시 융합합니다.
            base_docs = db.similarity_search(question, k=10)
            hyde_docs, _ = hyde_retrieve(question, k=10)
            retry_candidates = reciprocal_rank_fusion(
                [base_docs, hyde_docs],
                k=60,
                top_k=10,
            )
            retrieved_docs = rerank(
                question,
                retry_candidates,
                top_k=3,
            )
            if verbose:
                print("[4] NOT_SUPPORTED → HyDE 확장 검색 후 재시도")

    return answer, retrieved_docs


ans_sr, ctx_sr = self_rag(TEST_Q)
print("\n=== Self-RAG 최종 답변 ===")
print(ans_sr)


[1] Retrieve 필요? → YES
[3] 시도 1 — 자가 비평: NOT_SUPPORTED
[4] NOT_SUPPORTED → HyDE 확장 검색 후 재시도
[3] 시도 2 — 자가 비평: NOT_SUPPORTED

=== Self-RAG 최종 답변 ===
대중교통체계입니다.


## Step 6 : RAGAS 평가용 데이터셋 만들기

RAGAS 는 네 가지 자료가 필요합니다.
- `user_input` — 사용자 질문
- `response`   — RAG 가 생성한 답변
- `retrieved_contexts` — RAG 가 참고한 문서들
- `reference`  — 모범 답안 (Ground Truth)

**KorQuAD 는 사람이 작성한 정답이 데이터셋에 이미 포함**되어 있어, `reference` 를 따로 작성할 필요 없이 그대로 가져다 씁니다. 같은 질문 셋을 **Naive RAG** 와 **Advanced RAG** 두 가지로 풀고 결과를 비교합니다.

토큰 비용 통제를 위해 평가 질문은 5개만 사용합니다. (늘리려면 `EVAL_N` 변경)

In [ ]:
# 평가용 질문/정답 자동 추출 (KorQuAD)
EVAL_N = 20
eval_samples = list(raw_ds)[:EVAL_N]
questions = [example["question"] for example in eval_samples]
ground_truths = [example["answers"]["text"][0] for example in eval_samples]

# Naive RAG: 단일 질문 similarity top-3
naive_answers, naive_contexts = [], []
for index, question in enumerate(questions, start=1):
    contexts = naive_retriever.invoke(question)
    answer = (RAG_PROMPT | llm | StrOutputParser()).invoke(
        {"context": format_docs(contexts), "question": question}
    )
    naive_answers.append(answer)
    naive_contexts.append([doc.page_content for doc in contexts])
    print(f"[KorQuAD Naive] {index}/{EVAL_N}")

# Advanced RAG: Multi-Query → RRF → HyDE → Reranking → Self-RAG
adv_answers, adv_contexts = [], []
for index, question in enumerate(questions, start=1):
    answer, contexts = self_rag(question, max_retries=1, verbose=False)
    adv_answers.append(answer)
    adv_contexts.append([doc.page_content for doc in contexts])
    print(f"[KorQuAD Advanced] {index}/{EVAL_N}")

print(f"데이터셋 준비 완료 — {EVAL_N}개 질문 × 2개 파이프라인")


[KorQuAD Naive] 1/20
[KorQuAD Naive] 2/20
[KorQuAD Naive] 3/20
[KorQuAD Naive] 4/20
[KorQuAD Naive] 5/20
[KorQuAD Naive] 6/20
[KorQuAD Naive] 7/20
[KorQuAD Naive] 8/20
[KorQuAD Naive] 9/20
[KorQuAD Naive] 10/20
[KorQuAD Naive] 11/20
[KorQuAD Naive] 12/20
[KorQuAD Naive] 13/20
[KorQuAD Naive] 14/20
[KorQuAD Naive] 15/20
[KorQuAD Naive] 16/20
[KorQuAD Naive] 17/20
[KorQuAD Naive] 18/20
[KorQuAD Naive] 19/20
[KorQuAD Naive] 20/20
[KorQuAD Advanced] 1/20
[KorQuAD Advanced] 2/20
[KorQuAD Advanced] 3/20
[KorQuAD Advanced] 4/20
[KorQuAD Advanced] 5/20
[KorQuAD Advanced] 6/20
[KorQuAD Advanced] 7/20
[KorQuAD Advanced] 8/20
[KorQuAD Advanced] 9/20
[KorQuAD Advanced] 10/20
[KorQuAD Advanced] 11/20
[KorQuAD Advanced] 12/20
[KorQuAD Advanced] 13/20
[KorQuAD Advanced] 14/20
[KorQuAD Advanced] 15/20
[KorQuAD Advanced] 16/20
[KorQuAD Advanced] 17/20
[KorQuAD Advanced] 18/20
[KorQuAD Advanced] 19/20
[KorQuAD Advanced] 20/20
데이터셋 준비 완료 — 20개 질문 × 2개 파이프라인


In [ ]:
from datasets import Dataset

def make_dataset(answers, contexts):
    return Dataset.from_dict({
        "user_input":         questions,
        "response":           answers,
        "retrieved_contexts": contexts,
        "reference":          ground_truths,
    })

naive_ds = make_dataset(naive_answers, naive_contexts)
adv_ds   = make_dataset(adv_answers,   adv_contexts)

## Step 7 : RAGAS로 4대 지표 계산하기  

Judge LLM은 `gpt-4o-mini`로, 임베딩은 `text-embedding-3-small`로 설정합니다.  
(Judge에 더 강한 모델을 쓰면 채점은 더 정교해지지만 비용이 늘어납니다.)

In [ ]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness, answer_relevancy,
    context_precision, context_recall,
)

judge_llm  = ChatOpenAI(model="gpt-4o-mini", temperature=0)
judge_emb  = OpenAIEmbeddings(model="text-embedding-3-small")
metrics    = [faithfulness, answer_relevancy,
              context_precision, context_recall]

print("=== Naive RAG 채점 ===")
naive_result = evaluate(naive_ds, metrics=metrics,
                        llm=judge_llm, embeddings=judge_emb,
                        raise_exceptions=False)

print("=== Advanced RAG 채점 ===")
adv_result = evaluate(adv_ds, metrics=metrics,
                      llm=judge_llm, embeddings=judge_emb,
                      raise_exceptions=False)

=== Naive RAG 채점 ===


Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

=== Advanced RAG 채점 ===


Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', None)

naive_df = naive_result.to_pandas()
adv_df   = adv_result.to_pandas()

def summary(df, label):
    cols = ["faithfulness", "answer_relevancy",
            "context_precision", "context_recall"]
    avg = df[cols].mean()
    avg.name = label
    return avg

compare = pd.concat([summary(naive_df, "Naive RAG"),
                     summary(adv_df,   "Advanced RAG")], axis=1)
print(compare.round(3))
print("\nDelta (Advanced - Naive):")
print((compare["Advanced RAG"] - compare["Naive RAG"]).round(3))

                   Naive RAG  Advanced RAG
faithfulness           0.750         0.804
answer_relevancy       0.277         0.267
context_precision      0.733         0.800
context_recall         0.800         0.800

Delta (Advanced - Naive):
faithfulness         0.054
answer_relevancy    -0.011
context_precision    0.067
context_recall       0.000
dtype: float64


### 결과 해석 가이드

위 비교표를 처음 보면 **‘Advanced 가 더 나쁜 거 아닌가?’** 라는 착각을 하기 쉽습니다. KorQuAD 위에서의 결과 해석 방법을 정리합니다.

**1. `context_precision` 의 개선 (+) 이 Advanced RAG 의 핵심 효과**
검색 결과의 ‘상단’에 정답 문단을 두는 일을 Reranker 가 잘 했다는 의미. Δ가 0.05~0.15 정도면 잘 작동.

**2. `context_recall = 1.0` 으로 포화될 수 있다**
unique context 가 800개 정도면 Naive top-3 에도 정답이 거의 항상 들어옵니다. 이 지표는 더 큰 DB(수만 문서)에서 차이가 드러납니다.

**3. `faithfulness` 가 살짝 떨어질 수 있다**
Reranker 가 컨텍스트를 ‘짧고 집중’ 시키면 LLM이 그 좁은 정보에서 답을 만들 때 일부 주장이 “미뒷받침” 으로 채점되어 점수가 약간 내려갈 수 있음. **정상 범위 (-0.1 이내)**.

**4. `answer_relevancy` 가 0.2~0.4 로 낮은 이유 — KorQuAD 의 구조적 특성**
KorQuAD 정답은 *‘대중교통체계’* 같이 한 단어~한 구절. RAG 답변도 짧게 나오는데, RAGAS 의 `answer_relevancy` 는 **답변에서 질문을 역추론**해 원래 질문과의 유사도를 계산합니다. 답변이 한 단어면 역추론이 흐려져 점수가 낮아집니다. **모델 잘못이 아닌 데이터셋 특성**.

**5. 표본 20개로도 Δ가 ±0.05 이내면 ‘차이 없음’으로 봐야 한다**
20문항에서 ±0.05 는 표본 noise. 더 확실한 판단이 필요하면 `scipy.stats.ttest_rel` 로 통계 검정을 하거나 50~100문항으로 늘려야 합니다.

**6. 한국어 짧은 정답 벤치마크의 한계**
KorQuAD/KLUE-MRC 처럼 정답이 짧은 extractive QA 벤치마크는 `context_precision` 위주로 평가 효과를 봐야 하고, `answer_relevancy` 는 절대값보다 **Naive 대비 상대 변화**로 읽어야 합니다.

### Quiz  
위 표에서 Advanced RAG가 가장 크게 개선한 지표는 무엇인가요? 그리고 그 지표는 우리가 적용한 **어떤 기법**과 가장 직접적으로 연결될까요?  

**Answer (예시)**:  
보통 `context_precision`이 가장 크게 오릅니다. 이는 우리가 추가한 **Reranker**가 ‘진짜 관련도가 높은 문서를 상위에 두는 일’을 잘 했다는 의미입니다.  `context_recall`은 **Multi-Query**가 검색 폭을 넓혔다면 같이 오릅니다.  `faithfulness`와 `answer_relevancy`는 컨텍스트 품질이 올라가면 부수적으로 개선됩니다.

## Step 8 : (선택) 평가 데이터를 LLM으로 자동 생성하기  

현업에서는 모범 답안(`reference`)을 사람이 직접 작성하는 게 가장 큰 부담입니다.  
RAGAS는 **원본 문서만 주면 Question·Reference·Context 한 세트를 자동으로 만들어 주는** 기능을 제공합니다.  
자세한 사용법은 공식 문서를 참고하세요.

https://docs.ragas.io/en/stable/getstarted/rag_testset_generation/

---
# 추가 실습 — KLUE-MRC 한국어 뉴스 MRC 벤치마크로 RAG 평가하기

메인 실습은 위키 기반 **KorQuAD v1** 으로 진행했습니다. 이번 추가 실습은 도메인을 바꿔, **한국어 뉴스 기사 기반의 MRC 벤치마크 KLUE-MRC** 위에서 같은 파이프라인을 처음부터 다시 조립해 봅니다.

**KLUE-MRC**
- 카카오·네이버 등 한국 NLP 팀이 함께 만든 한국어 표준 벤치마크 KLUE 의 MRC 태스크
- 한국어 **뉴스 기사** 기반 (KorQuAD 의 위키와 도메인이 다름)
- 사람이 직접 작성한 정답 포함
- **`is_impossible=True`** 인 답할 수 없는 질문도 일부 포함 → 데이터 필터링이 필요한 도전적 케이스

위키 기반 KorQuAD 와 비교했을 때 어떤 차이(질문 스타일, 검색 난이도, 점수 분포)가 나는지 직접 관찰해 보세요.

이번에도 일부만 샘플링해서 토큰 비용을 통제합니다.
- Vector DB 에 들어갈 unique context: 약 200개
- 평가 질문: 20개
- 예상 비용: GPT-4o-mini 기준 RAGAS 평가까지 합쳐서 약 \$0.10 ~ \$0.20

**📥 데이터셋 다운로드 / 출처**
- HuggingFace `datasets` 자동 다운로드: <https://huggingface.co/datasets/klue>
- KLUE 공식 사이트: <https://klue-benchmark.com/>
- KLUE 논문: <https://arxiv.org/abs/2105.09680>

> 다른 데이터셋으로 한 번 더 해보고 싶다면:  
> - MIRACL 한국어: <https://huggingface.co/datasets/miracl/miracl> (config: `ko`)  
> - 영어 SQuAD: <https://huggingface.co/datasets/rajpurkar/squad>

### Step A. 데이터셋 로드

`datasets` 라이브러리로 KLUE-MRC 를 한 줄에 받아옵니다. KLUE 는 여러 sub-task 가 있는 멀티태스크 벤치마크라서 config 이름 `"mrc"` 를 명시해야 합니다.

데이터셋 페이지: <https://huggingface.co/datasets/klue>

In [ ]:
from datasets import load_dataset

ds_klue = load_dataset("klue", "mrc", split="validation")
print(ds_klue)
print("\n--- 샘플 1건 ---")
print({k: ds_klue[0][k] for k in ds_klue.column_names})

README.md: 0.00B [00:00, ?B/s]

mrc/train-00000-of-00001.parquet:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

mrc/validation-00000-of-00001.parquet:   0%|          | 0.00/8.68M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/17554 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5841 [00:00<?, ? examples/s]

Dataset({
    features: ['title', 'context', 'news_category', 'source', 'guid', 'is_impossible', 'question_type', 'question', 'answers'],
    num_rows: 5841
})

--- 샘플 1건 ---
{'title': 'BMW 코리아, 창립 25주년 기념 ‘BMW 코리아 25주년 에디션’ 한정 출시', 'context': 'BMW 코리아(대표 한상윤)는 창립 25주년을 기념하는 ‘BMW 코리아 25주년 에디션’을 한정 출시한다고 밝혔다. 이번 BMW 코리아 25주년 에디션(이하 25주년 에디션)은 BMW 3시리즈와 5시리즈, 7시리즈, 8시리즈 총 4종, 6개 모델로 출시되며, BMW 클래식 모델들로 선보인 바 있는 헤리티지 컬러가 차체에 적용돼 레트로한 느낌과 신구의 조화가 어우러진 차별화된 매력을 자랑한다. 먼저 뉴 320i 및 뉴 320d 25주년 에디션은 트림에 따라 옥스포드 그린(50대 한정) 또는 마카오 블루(50대 한정) 컬러가 적용된다. 럭셔리 라인에 적용되는 옥스포드 그린은 지난 1999년 3세대 3시리즈를 통해 처음 선보인 색상으로 짙은 녹색과 풍부한 펄이 오묘한 조화를 이루는 것이 특징이다. M 스포츠 패키지 트림에 적용되는 마카오 블루는 1988년 2세대 3시리즈를 통해 처음 선보인 바 있으며, 보랏빛 감도는 컬러감이 매력이다. 뉴 520d 25주년 에디션(25대 한정)은 프로즌 브릴리언트 화이트 컬러로 출시된다. BMW가 2011년에 처음 선보인 프로즌 브릴리언트 화이트는 한층 더 환하고 깊은 색감을 자랑하며, 특히 표면을 무광으로 마감해 특별함을 더했다. 뉴 530i 25주년 에디션(25대 한정)은 뉴 3시리즈 25주년 에디션에도 적용된 마카오 블루 컬러가 조합된다. 뉴 740Li 25주년 에디션(7대 한정)에는 말라카이트 그린 다크 색상이 적용된다. 잔잔하면서도 오묘한 깊은 녹색을 발산하는 말라카이트 그린 다크는 장식재로 

### Step B. Context 추출 + 중복 제거 (+ is_impossible 필터링)

KLUE-MRC 에는 KorQuAD 에는 없는 **`is_impossible=True`** 케이스가 섞여 있습니다 (= context 만 보고는 답할 수 없는 질문). 평가용 ground_truth 가 비어 있으면 RAGAS 의 `context_recall` 이 깨지므로, 답이 있는 샘플만 남기세요.

- `ds_klue.filter(lambda x: not x["is_impossible"])` 로 답 있는 것만 추리고
- `shuffle(seed=42).select(range(300))` 으로 300개 샘플링
- 그 중 `context` 필드 기준으로 중복 제거 (보통 150~200개)
- 각각을 `Document(page_content=..., metadata={"title": ex["title"]})` 로 감싸 `context_docs` 에 담기

In [ ]:
# 답이 있는 KLUE-MRC 샘플만 선택하고 context를 중복 제거합니다.
from langchain_core.documents import Document

answerable_klue = (
    ds_klue
    .filter(lambda example: not example["is_impossible"])
    .shuffle(seed=42)
    .select(range(300))
)

unique_contexts_klue = {}
for example in answerable_klue:
    context = example["context"]
    if context not in unique_contexts_klue:
        unique_contexts_klue[context] = example.get("title", "")

context_docs_klue = [
    Document(
        page_content=context,
        metadata={"title": title, "domain": "KLUE-MRC"},
    )
    for context, title in unique_contexts_klue.items()
]

# KorQuAD와 같은 splitter를 사용해 도메인만 바꿔 비교합니다.
docs_klue = splitter.split_documents(context_docs_klue)

print("answerable samples:", len(answerable_klue))
print("unique contexts:", len(context_docs_klue))
print("chunks:", len(docs_klue))
print(context_docs_klue[0].page_content[:200])


Filter:   0%|          | 0/5841 [00:00<?, ? examples/s]

answerable samples: 300
unique contexts: 299
chunks: 863
국내 가상화폐 시장에서 해킹, 다단계 사기 등이 극성을 부리고 있다.비트코인(사진)에 이어 세계 2위 규모(자산총액 3000억원)의 가상화폐인 ‘리플’의 대규모 해킹 사건이 국내에서 발생한 것으로 9일 확인됐다. 리플은 운영 주체가 명확하지 않은 비트코인과 달리 미국 리플랩스라는 회사에서 운영하는 가상화폐다.국내 거래소인 ‘디지털게이트코리아’ 회원 등 20


### Step C. Embedding + VectorStore

메인 실습에서 만든 `embedding` (`OpenAIEmbeddings(model="text-embedding-3-small")`) 을 그대로 재사용해, `context_docs` 로 새 Chroma DB `db_klue` 를 만드세요. (메인 실습의 `db` 변수를 덮어쓰지 마세요. 비교가 안 됩니다.)

> ⚠️ **batch 적재 필수** — KLUE-MRC 의 뉴스 context 는 평균 토큰 수가 커서, 150개 이상을 한 번에 `Chroma.from_documents` 로 넘기면 OpenAI embeddings 의 **300k 토큰/요청 한도** 에 걸려 `BadRequestError` 가 납니다. 메인 cell 8 처럼 100개씩 batch 로 `add_documents` 호출하세요:
> ```python
> db_klue = Chroma(embedding_function=embedding)
> BATCH = 100
> for i in range(0, len(context_docs), BATCH):
>     db_klue.add_documents(context_docs[i:i+BATCH])
> ```

> 인덱싱 토큰 비용: 약 200개 context × 평균 600 토큰 ≈ **120k 토큰** (≈ \$0.003)

In [ ]:
# KorQuAD와 섞이지 않도록 별도의 collection 이름을 사용합니다.
db_klue = Chroma(
    collection_name="klue_mrc",
    embedding_function=embedding,
)

BATCH_KLUE = 100
for start in range(0, len(docs_klue), BATCH_KLUE):
    batch = docs_klue[start:start + BATCH_KLUE]
    db_klue.add_documents(batch)
    print(
        f"KLUE 적재: {min(start + BATCH_KLUE, len(docs_klue))}"
        f"/{len(docs_klue)}"
    )

print("KLUE-MRC VectorStore 준비 완료")


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


KLUE 적재: 100/863
KLUE 적재: 200/863
KLUE 적재: 300/863
KLUE 적재: 400/863
KLUE 적재: 500/863
KLUE 적재: 600/863
KLUE 적재: 700/863
KLUE 적재: 800/863
KLUE 적재: 863/863
KLUE-MRC VectorStore 준비 완료


### Step D. 평가용 질문/정답 세트 추출

Step B 에서 필터링·샘플링한 데이터 중 **앞에서 20개**를 평가용으로 떼어내세요.

- `questions_klue` : 각 샘플의 `question` 필드 (문자열 20개)
- `ground_truths_klue` : 각 샘플의 `answers["text"][0]` (정답이 여러 개일 경우 첫 번째 사용)

> 참고: KLUE-MRC 는 정답이 한 구절~한 문장 단위의 **extractive QA** 입니다. 짧은 정답은 RAGAS 의 `context_recall` 변동성을 키우는 경향이 있으니, 평균을 함께 봐주세요.

In [ ]:
EVAL_N_KLUE = 20
eval_samples_klue = list(answerable_klue)[:EVAL_N_KLUE]

questions_klue = [
    example["question"]
    for example in eval_samples_klue
]
ground_truths_klue = [
    example["answers"]["text"][0]
    for example in eval_samples_klue
]

assert len(questions_klue) == EVAL_N_KLUE
assert len(ground_truths_klue) == EVAL_N_KLUE

print("질문 수:", len(questions_klue))
print("첫 질문:", questions_klue[0])
print("첫 정답:", ground_truths_klue[0])


질문 수: 20
첫 질문: 국내에서 해킹을 당한 리플이 들어간 통장의 갯수는?
첫 정답: 두 개


### Step E. Naive RAG 베이스라인 (KLUE)

메인 실습의 `RAG_PROMPT` 를 그대로 써도 되고, 뉴스 도메인 특성을 살려 *“기사 본문에 근거해서만 답하세요”* 같은 지시를 추가해도 좋습니다.

- `naive_retriever_klue = db_klue.as_retriever(search_type="similarity", search_kwargs={"k": 3})`
- 체인 구조는 메인 Step 1 과 동일

In [ ]:
RAG_PROMPT_KLUE = ChatPromptTemplate.from_template(
    "다음 뉴스 기사 본문만 근거로 질문에 한국어로 간결하게 답하세요. "
    "기사에 없는 내용은 만들지 마세요.\n\n"
    "[기사 본문]\n{context}\n\n"
    "[질문]\n{question}\n\n"
    "[답변]"
)

naive_retriever_klue = db_klue.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)

naive_chain_klue = (
    {
        "context": naive_retriever_klue | format_docs,
        "question": RunnablePassthrough(),
    }
    | RAG_PROMPT_KLUE
    | llm
    | StrOutputParser()
)

print("Q:", questions_klue[0])
print("A:", naive_chain_klue.invoke(questions_klue[0]))


Q: 국내에서 해킹을 당한 리플이 들어간 통장의 갯수는?


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


A: 200여 개의 계좌입니다.


### Step F. Multi-Query Retrieval

메인 Step 2 와 동일하게 `MultiQueryRetriever.from_llm(...)` 으로 KLUE 검색기를 감싸세요. 한국어 질문이 들어가면 gpt-4o-mini 가 한국어로 유사 질문을 만들어 줍니다.

확장 질문 로깅을 켜서 어떤 한국어 변형 질문이 만들어지는지 직접 눈으로 확인하세요.

In [ ]:
import logging
from langchain.retrievers.multi_query import MultiQueryRetriever

logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

multi_query_retriever_klue = MultiQueryRetriever.from_llm(
    retriever=db_klue.as_retriever(search_kwargs={"k": 3}),
    llm=ChatOpenAI(model="gpt-4o-mini", temperature=0),
)

docs_mq_klue = multi_query_retriever_klue.invoke(questions_klue[0])
print("Multi-Query 검색 문서 수:", len(docs_mq_klue))
print(docs_mq_klue[0].page_content[:300])


INFO:langchain.retrievers.multi_query:Generated queries: ['1. 국내에서 해킹으로 피해를 입은 리플이 포함된 통장의 수는 몇 개인가요?  ', '2. 한국에서 해킹 사건에 연루된 리플이 있는 통장 수는 얼마인가요?  ', '3. 국내에서 해킹으로 영향을 받은 리플이 포함된 계좌의 개수는 어떻게 되나요?']


Multi-Query 검색 문서 수: 4
국내 가상화폐 시장에서 해킹, 다단계 사기 등이 극성을 부리고 있다.비트코인(사진)에 이어 세계 2위 규모(자산총액 3000억원)의 가상화폐인 ‘리플’의 대규모 해킹 사건이 국내에서 발생한 것으로 9일 확인됐다. 리플은 운영 주체가 명확하지 않은 비트코인과 달리 미국 리플랩스라는 회사에서 운영하는 가상화폐다.국내 거래소인 ‘디지털게이트코리아’ 회원 등 200여명의 계좌에서 3월20일부터 30일까지 3억원어치의 리플이 도난당했다. 국내에서 발생한 가상화폐 해킹 사건으로는 이례적인 규모다. 200여명의 계좌에서 빠져나간 리플은 두 개의


### Step G. HyDE 직접 구현

메인 Step 3 의 `HYDE_PROMPT` 를 그대로 써도 되고, 뉴스 도메인용으로 *“기자가 쓴 한 문단 형태”* 로 답하라는 지시를 추가해도 됩니다.

`hyde_retrieve_klue(question, k=3)` 함수를 만들고 `db_klue` 위에서 동작하도록 하세요.

In [ ]:
HYDE_PROMPT_KLUE = ChatPromptTemplate.from_template(
    "당신은 한국어 뉴스 기자입니다. 다음 질문의 답이 포함될 법한 기사 한 문단을 "
    "사실적인 문체로 작성하세요. 검색용 가상 문서이므로 한 문단만 출력하세요."
    "\n\n질문: {question}\n\n가상 기사:"
)

hyde_generator_klue = HYDE_PROMPT_KLUE | llm | StrOutputParser()

def hyde_retrieve_klue(question, k=3):
    hypothetical = hyde_generator_klue.invoke({"question": question})
    documents = db_klue.similarity_search(hypothetical, k=k)
    return documents, hypothetical


docs_hyde_klue, hyp_klue = hyde_retrieve_klue(questions_klue[0])
print("가상 기사:\n", hyp_klue[:300])
print("\n첫 검색 문서:\n", docs_hyde_klue[0].page_content[:300])


가상 기사:
 최근 국내에서 발생한 해킹 사건으로 인해 리플이 포함된 통장 계좌가 1,200개 이상 피해를 입은 것으로 확인됐다. 금융감독원은 해당 사건에 대한 조사를 진행 중이며, 피해자들에게는 즉각적인 대응을 권장하고 있다. 해킹으로 인한 피해를 최소화하기 위해 금융기관들은 보안 시스템을 강화하고, 고객들에게 비밀번호 변경 및 이중 인증 설정을 권장하고 있다. 이번 사건은 암호화폐와 관련된 금융 범죄의 위험성을 다시 한번 일깨우는 계기가 되고 있다.

첫 검색 문서:
 국내 가상화폐 시장에서 해킹, 다단계 사기 등이 극성을 부리고 있다.비트코인(사진)에 이어 세계 2위 규모(자산총액 3000억원)의 가상화폐인 ‘리플’의 대규모 해킹 사건이 국내에서 발생한 것으로 9일 확인됐다. 리플은 운영 주체가 명확하지 않은 비트코인과 달리 미국 리플랩스라는 회사에서 운영하는 가상화폐다.국내 거래소인 ‘디지털게이트코리아’ 회원 등 200여명의 계좌에서 3월20일부터 30일까지 3억원어치의 리플이 도난당했다. 국내에서 발생한 가상화폐 해킹 사건으로는 이례적인 규모다. 200여명의 계좌에서 빠져나간 리플은 두 개의


### Step H. Multilingual Cross-encoder Reranker

메인 Step 4 에서 이미 `BAAI/bge-reranker-v2-m3` 같은 다국어 reranker 를 사용하고 있습니다. 추가 실습에서는:

- 메인의 `reranker` 인스턴스를 그대로 재사용하거나
- 다른 다국어 reranker 와 비교해 봐도 좋습니다:
  - `Alibaba-NLP/gte-multilingual-reranker-base` — <https://huggingface.co/Alibaba-NLP/gte-multilingual-reranker-base>
  - `jinaai/jina-reranker-v2-base-multilingual` — <https://huggingface.co/jinaai/jina-reranker-v2-base-multilingual>

`rerank_klue(query, docs, top_k=3)` 함수를 만드세요. (메인 Step 4 의 `rerank` 와 동일 구조)

In [ ]:
# 같은 다국어 모델을 다시 다운로드하지 않고 메인 실습 인스턴스를 재사용합니다.
reranker_klue = reranker

def rerank_klue(query, documents, top_k=3):
    if not documents:
        return []

    pairs = [(query, document.page_content) for document in documents]
    scores = reranker_klue.predict(pairs)
    ranked = sorted(
        zip(documents, scores),
        key=lambda item: item[1],
        reverse=True,
    )
    return [document for document, _ in ranked[:top_k]]


candidates_klue = db_klue.similarity_search(questions_klue[0], k=10)
top3_klue = rerank_klue(questions_klue[0], candidates_klue, top_k=3)
print(f"후보 {len(candidates_klue)}개 → 최종 {len(top3_klue)}개")
print(top3_klue[0].page_content[:300])


후보 10개 → 최종 3개
국내 가상화폐 시장에서 해킹, 다단계 사기 등이 극성을 부리고 있다.비트코인(사진)에 이어 세계 2위 규모(자산총액 3000억원)의 가상화폐인 ‘리플’의 대규모 해킹 사건이 국내에서 발생한 것으로 9일 확인됐다. 리플은 운영 주체가 명확하지 않은 비트코인과 달리 미국 리플랩스라는 회사에서 운영하는 가상화폐다.국내 거래소인 ‘디지털게이트코리아’ 회원 등 200여명의 계좌에서 3월20일부터 30일까지 3억원어치의 리플이 도난당했다. 국내에서 발생한 가상화폐 해킹 사건으로는 이례적인 규모다. 200여명의 계좌에서 빠져나간 리플은 두 개의


### Step I. Advanced RAG 체인 (넓게 → Rerank → LLM)

메인 Step 5 흐름과 동일.
1. `db_klue.as_retriever(search_kwargs={"k": 10}).invoke(question)` 로 후보 10개
2. `rerank_klue(question, candidates, top_k=3)` 로 좁힘
3. `RAG_PROMPT` + `llm` 으로 답변 생성

함수가 `(answer, top_docs)` 둘 다 반환하도록 만들어 두면 다음 평가 단계에서 그대로 씁니다.

In [ ]:
def advanced_retrieve_klue(question, top_k=3):
    # KLUE: Multi-Query → RRF(+HyDE) → Cross-Encoder Reranking
    expanded_queries = [question, *fan_out_queries(question, n=4)]
    results_per_query = [
        db_klue.similarity_search(query, k=5)
        for query in expanded_queries
    ]

    hyde_docs, _ = hyde_retrieve_klue(question, k=5)
    results_per_query.append(hyde_docs)

    fused_candidates = reciprocal_rank_fusion(
        results_per_query,
        k=60,
        top_k=10,
    )
    return rerank_klue(question, fused_candidates, top_k=top_k)


def advanced_rag_klue(question):
    top_docs = advanced_retrieve_klue(question, top_k=3)
    answer = (RAG_PROMPT_KLUE | llm | StrOutputParser()).invoke(
        {"context": format_docs(top_docs), "question": question}
    )
    return answer, top_docs


def self_rag_klue(question, max_retries=1, verbose=True):
    decision = (
        RETRIEVE_DECISION_PROMPT | llm | StrOutputParser()
    ).invoke({"question": question}).strip().upper()

    if verbose:
        print(f"[1] Retrieve 필요? → {decision}")

    if decision.startswith("NO"):
        return llm.invoke(question).content, []

    retrieved_docs = advanced_retrieve_klue(question, top_k=3)

    for attempt in range(max_retries + 1):
        context = format_docs(retrieved_docs)
        answer = (RAG_PROMPT_KLUE | llm | StrOutputParser()).invoke(
            {"context": context, "question": question}
        )
        critique = (
            CRITIQUE_PROMPT | llm | StrOutputParser()
        ).invoke({"context": context, "answer": answer}).strip().upper()

        supported = (
            critique.startswith("SUPPORTED")
            and "NOT_SUPPORTED" not in critique
        )
        if verbose:
            print(f"[2] 시도 {attempt + 1} — 자가 비평: {critique}")

        if supported:
            return answer, retrieved_docs

        if attempt < max_retries:
            base_docs = db_klue.similarity_search(question, k=10)
            hyde_docs, _ = hyde_retrieve_klue(question, k=10)
            retry_candidates = reciprocal_rank_fusion(
                [base_docs, hyde_docs],
                k=60,
                top_k=10,
            )
            retrieved_docs = rerank_klue(
                question,
                retry_candidates,
                top_k=3,
            )

    return answer, retrieved_docs


answer_klue, contexts_klue = self_rag_klue(questions_klue[0])
print("\nAdvanced KLUE 답변:\n", answer_klue)
print("최종 컨텍스트 수:", len(contexts_klue))


[1] Retrieve 필요? → YES
[2] 시도 1 — 자가 비평: SUPPORTED

Advanced KLUE 답변:
 200여 개의 계좌입니다.
최종 컨텍스트 수: 3


### Step J. RAGAS 로 Naive vs Advanced 비교

메인 Step 6/7 흐름을 KLUE-MRC 변수(`_klue`) 로 옮겨 동일하게 수행하세요.

1. 20개 질문 각각을 Naive / Advanced 파이프라인에 돌려 답변과 컨텍스트 수집
2. `Dataset.from_dict({...})` 로 `naive_ds_klue`, `adv_ds_klue` 두 개 생성 (키: `user_input / response / retrieved_contexts / reference`)
3. `evaluate(..., metrics=[faithfulness, answer_relevancy, context_precision, context_recall], llm=judge_llm, embeddings=judge_emb, raise_exceptions=False)` 두 번
4. 평균표로 비교

메인의 KorQuAD 결과와 점수가 어떻게 다른지 옆에 같이 적어두면 학습 효과가 큽니다.

In [ ]:
# 1) 같은 KLUE 질문을 Naive와 Advanced 파이프라인으로 각각 처리합니다.
naive_answers_klue, naive_contexts_klue = [], []
adv_answers_klue, adv_contexts_klue = [], []

for index, question in enumerate(questions_klue, start=1):
    naive_docs_klue = naive_retriever_klue.invoke(question)
    naive_answer_klue = (RAG_PROMPT_KLUE | llm | StrOutputParser()).invoke(
        {"context": format_docs(naive_docs_klue), "question": question}
    )
    naive_answers_klue.append(naive_answer_klue)
    naive_contexts_klue.append(
        [document.page_content for document in naive_docs_klue]
    )
    print(f"[KLUE Naive] {index}/{EVAL_N_KLUE}")

for index, question in enumerate(questions_klue, start=1):
    advanced_answer_klue, advanced_docs_klue = self_rag_klue(
        question,
        max_retries=1,
        verbose=False,
    )
    adv_answers_klue.append(advanced_answer_klue)
    adv_contexts_klue.append(
        [document.page_content for document in advanced_docs_klue]
    )
    print(f"[KLUE Advanced] {index}/{EVAL_N_KLUE}")


# 2) RAGAS 형식으로 변환합니다.
def make_dataset_klue(answers, contexts):
    return Dataset.from_dict({
        "user_input": questions_klue,
        "response": answers,
        "retrieved_contexts": contexts,
        "reference": ground_truths_klue,
    })


naive_ds_klue = make_dataset_klue(
    naive_answers_klue,
    naive_contexts_klue,
)
adv_ds_klue = make_dataset_klue(
    adv_answers_klue,
    adv_contexts_klue,
)

# 3) KorQuAD와 같은 Judge와 지표로 평가합니다.
print("=== KLUE Naive RAG 채점 ===")
naive_result_klue = evaluate(
    naive_ds_klue,
    metrics=metrics,
    llm=judge_llm,
    embeddings=judge_emb,
    raise_exceptions=False,
)

print("=== KLUE Advanced RAG 채점 ===")
adv_result_klue = evaluate(
    adv_ds_klue,
    metrics=metrics,
    llm=judge_llm,
    embeddings=judge_emb,
    raise_exceptions=False,
)

naive_df_klue = naive_result_klue.to_pandas()
adv_df_klue = adv_result_klue.to_pandas()

compare_klue = pd.concat(
    [
        summary(naive_df_klue, "Naive RAG"),
        summary(adv_df_klue, "Advanced RAG"),
    ],
    axis=1,
)

print("\n=== KLUE-MRC Naive vs Advanced ===")
print(compare_klue.round(3))

delta_klue = compare_klue["Advanced RAG"] - compare_klue["Naive RAG"]
print("\nDelta (Advanced - Naive):")
print(delta_klue.round(3))

# 4) 두 도메인의 표를 같은 형태로 나란히 봅니다.
compare_domains = pd.concat(
    {
        "KorQuAD(위키)": compare.T,
        "KLUE-MRC(뉴스)": compare_klue.T,
    },
    names=["domain", "pipeline"],
)

print("\n=== 도메인별 전체 비교 ===")
print(compare_domains.round(3))

advanced_domain_delta = (
    compare_klue["Advanced RAG"] - compare["Advanced RAG"]
)
print("\nAdvanced 기준 도메인 차이 (KLUE - KorQuAD):")
print(advanced_domain_delta.round(3))
print(
    "가장 크게 달라진 지표:",
    advanced_domain_delta.abs().idxmax(),
)


[KLUE Naive] 1/20
[KLUE Naive] 2/20
[KLUE Naive] 3/20
[KLUE Naive] 4/20
[KLUE Naive] 5/20
[KLUE Naive] 6/20
[KLUE Naive] 7/20
[KLUE Naive] 8/20
[KLUE Naive] 9/20
[KLUE Naive] 10/20
[KLUE Naive] 11/20
[KLUE Naive] 12/20
[KLUE Naive] 13/20
[KLUE Naive] 14/20
[KLUE Naive] 15/20
[KLUE Naive] 16/20
[KLUE Naive] 17/20
[KLUE Naive] 18/20
[KLUE Naive] 19/20
[KLUE Naive] 20/20
[KLUE Advanced] 1/20
[KLUE Advanced] 2/20
[KLUE Advanced] 3/20
[KLUE Advanced] 4/20
[KLUE Advanced] 5/20
[KLUE Advanced] 6/20
[KLUE Advanced] 7/20
[KLUE Advanced] 8/20
[KLUE Advanced] 9/20
[KLUE Advanced] 10/20
[KLUE Advanced] 11/20
[KLUE Advanced] 12/20
[KLUE Advanced] 13/20
[KLUE Advanced] 14/20
[KLUE Advanced] 15/20
[KLUE Advanced] 16/20
[KLUE Advanced] 17/20
[KLUE Advanced] 18/20
[KLUE Advanced] 19/20
[KLUE Advanced] 20/20
=== KLUE Naive RAG 채점 ===


Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

=== KLUE Advanced RAG 채점 ===


Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]


=== KLUE-MRC Naive vs Advanced ===
                   Naive RAG  Advanced RAG
faithfulness           0.650         0.653
answer_relevancy       0.252         0.293
context_precision      0.642         0.892
context_recall         0.750         0.800

Delta (Advanced - Naive):
faithfulness         0.003
answer_relevancy     0.041
context_precision    0.250
context_recall       0.050
dtype: float64

=== 도메인별 전체 비교 ===
                           faithfulness  answer_relevancy  context_precision  \
domain       pipeline                                                          
KorQuAD(위키)  Naive RAG            0.750             0.277              0.733   
             Advanced RAG         0.804             0.267              0.800   
KLUE-MRC(뉴스) Naive RAG            0.650             0.252              0.642   
             Advanced RAG         0.653             0.293              0.892   

                           context_recall  
domain       pipeline                      
KorQuAD(위키

### Step K. (선택) 좀 더 큰 샘플로 통계적 신뢰도 확보

질문 20개로는 표본 분산이 커서 Naive vs Advanced 차이가 우연일 수도 있습니다. 토큰 비용이 허용된다면 50~100문항으로 늘려 paired t-test 같은 간단한 통계 검정으로 차이가 유의한지 확인해 보세요.

참고: `scipy.stats.ttest_rel(naive_df["faithfulness"], adv_df["faithfulness"])`

In [ ]:
# TODO (선택): 질문 수를 늘려 같은 평가를 반복한 뒤 paired t-test 로 차이 검정



In [ ]:
# TODO 차이 검정
from scipy import stats
import pandas as pd

# 검정할 평가 지표 목록
metrics = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]

print("=== Paired t-test 결과 (Advanced vs Naive) ===")
for metric in metrics:
    # NaN(결측치)가 발생할 수 있으므로, 두 데이터 모두 존재하는 행만 필터링
    valid_data = pd.concat([naive_df[metric], adv_df[metric]], axis=1).dropna()

    # 길이가 2 미만이면 t-test가 불가능하므로 예외 처리
    if len(valid_data) < 2:
        print(f"[{metric}] 유효한 데이터가 부족하여 검정 불가\n")
        continue

    naive_scores = valid_data.iloc[:, 0]
    adv_scores = valid_data.iloc[:, 1]

    # Paired t-test 수행 (Advanced - Naive)
    t_stat, p_value = stats.ttest_rel(adv_scores, naive_scores)

    print(f"[{metric}]")
    print(f" - t-statistic : {t_stat:.4f}")
    print(f" - p-value     : {p_value:.4f}")

    # 유의수준 0.05 기준 판별
    if p_value < 0.05:
        print(" -> 결론: 두 RAG 모델 간에 통계적으로 유의미한 차이가 **있습니다**.\n")
    else:
        print(" -> 결론: 두 RAG 모델 간에 통계적으로 유의미한 차이가 **없습니다**.\n")

=== Paired t-test 결과 (Advanced vs Naive) ===
[faithfulness]
 - t-statistic : 0.4764
 - p-value     : 0.6392
 -> 결론: 두 RAG 모델 간에 통계적으로 유의미한 차이가 **없습니다**.

[answer_relevancy]
 - t-statistic : -0.3515
 - p-value     : 0.7291
 -> 결론: 두 RAG 모델 간에 통계적으로 유의미한 차이가 **없습니다**.

[context_precision]
 - t-statistic : 0.6037
 - p-value     : 0.5532
 -> 결론: 두 RAG 모델 간에 통계적으로 유의미한 차이가 **없습니다**.

[context_recall]
 - t-statistic : 0.0000
 - p-value     : 1.0000
 -> 결론: 두 RAG 모델 간에 통계적으로 유의미한 차이가 **없습니다**.



### 마지막 Quiz — 결과표를 근거로 작성하는 방법

아래 문장에서 대괄호 부분을 실제 실행 결과로 교체하세요. 결과가 나오기 전에 수치를 추측해서 쓰면 안 됩니다.

1. **도메인 비교**  
   Advanced RAG 기준으로 KorQuAD와 KLUE-MRC 사이에서 가장 크게 달라진 지표는
   `[advanced_domain_delta.abs().idxmax() 결과]`였다. KorQuAD는 비교적 설명적이고 구조화된 위키 문장이 많은 반면,
   KLUE-MRC 뉴스는 날짜·수치·고유명사·인용 관계가 조밀하고 사건별 표현이 다양하다. 이 때문에 짧은 정답을 포함한
   정확한 기사 Chunk를 찾고, 여러 유사 문단 중 정답 근거를 위쪽에 배치하는 난도가 달라질 수 있다.

2. **Advanced 효과**  
   KLUE에서 가장 크게 개선된 지표는 `[delta_klue.idxmax() 결과]`이고 개선량은 `[해당 값]`이다.
   `context_precision` 향상은 주로 Cross-Encoder reranker와 연결하고, `context_recall` 향상은 Multi-Query·RRF·HyDE가
   검색 폭을 넓힌 효과와 연결해서 해석한다. `faithfulness`와 `answer_relevancy`는 검색 문맥의 품질 및 Self-RAG 비평의
   영향을 함께 받는다.

3. **`is_impossible` 케이스**  
   정답이 존재하지 않는 샘플을 일반 정답형 평가에 그대로 섞으면 `reference`를 근거로 계산하는
   `context_recall`이 가장 먼저 정의 불가능하거나 `NaN`에 가까운 상태로 망가질 수 있다. 모델이 억지 답변을 만들면
   `faithfulness`도 함께 낮아질 수 있다. 답변 불가 질문은 별도 거부 정확도(abstention accuracy)로 평가하는 편이 적절하다.

4. **결론 작성 규칙**  
   “Advanced가 항상 우수하다”라고 쓰지 말고, 실제 Delta가 양수인 지표와 음수인 지표를 모두 언급한다.
   Multi-Query·HyDE·Self-RAG는 LLM 호출 수와 지연 시간을 늘리므로, 품질 향상이 작다면 비용 대비 효과도 함께 논의한다.


## 회고  

이번 실습을 통해 가장 기본이 되는 Naive RAG에서 출발하여, 성능을 끌어올리기 위한 Advanced·Modular RAG 파이프라인을 직접 단계별로 구현해 보았습니다.  

- __검색 성능 강화 (Pre/Post-retrieval)__: 질문을 확장하는 Multi-Query, 검색 결과를 융합하는 RAG-Fusion(RRF), 가상의 정답을 생성해 검색하는 HyDE, 그리고 Cross-Encoder 기반의 Reranking 기술을 결합하여 검색의 정밀도를 높이는 과정을 실습했습니다.  

- __Self-RAG 도입__: LLM이 스스로 검색 필요성을 판단하고 생성된 답변을 자가 비평(Critique)하여, 필요시 다시 검색을 수행하도록 만드는 로직을 뼈대부터 구현해 보며 모델의 환각(Hallucination)을 줄이는 방법을 배웠습니다.  

- __RAGAS 기반 정량적 평가__: 눈대중이 아닌 RAGAS 프레임워크를 활용해 Faithfulness, Answer Relevancy, Context Precision, Context Recall의 4대 지표로 파이프라인의 성능 향상(Naive vs Advanced)을 숫자로 직접 확인하고 통계적 유의미성까지 점검해 볼 수 있었습니다.  

- __도메인 특성 이해__: 위키피디아 기반의 KorQuAD와 뉴스 기사 기반의 KLUE-MRC 데이터셋을 비교하며, 도메인(문체, 정답의 길이 등)과 is_impossible 데이터 여부에 따라 RAG 평가 지표가 어떻게 달라지는지 분석하는 시각을 길렀습니다.  

RAG의 이론을 학습하고 과제를 수행하면서 LLM의 도움이 없었다면 불가능했을거라 생각한다.  
아직까지도 코드를 빠르게 이해하고 따라갈순없지만 제주도를 다녀오고 다른 그루분들의 조언과 팁으로 전보단 코드가 어떻게 흘러가는지 이해하게 되었고, 이론이랑 코드랑 연결지어 학습할 수 있도록 성장하게 된거같다. 지식을 나누준 많은 그루분들께 감사하다.